# Feature Engineering

This notebook builds a robust feature set from each 10-value sequence and prepares the data for downstream modeling and evaluation.

It covers validation, feature extraction, quality checks, and final export to a model-ready dataset.

## 1. Load and validate the dataset

The first step loads the cleaned dataset and verifies that each required sequence column and label are present before feature engineering begins.

In [27]:

import pandas as pd
import numpy as np

# File path
data_dir = Path(r"D:\personal projects\humanVsComputerGeneration\Data")
file_path = data_dir / "clean_combined_dataset.csv"

# Load data
df = pd.read_csv(file_path)


## 2. Feature extraction

This section defines the feature-generation logic for each sequence, including summary statistics, repetition patterns, direction changes, and range counts.

In [28]:

# Identify the 10 sequence columns
sequence_cols = [f"number_{i}" for i in range(1, 11)]

# Validate columns
missing_cols = [c for c in sequence_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

if "label" not in df.columns:
    raise ValueError("The dataset must contain a 'label' column.")

# Ensure sequence values are numeric
df[sequence_cols] = df[sequence_cols].apply(
    pd.to_numeric, errors="raise"
)

print("Sequence columns:", sequence_cols)
print("Number of positions:", len(sequence_cols))
print("Labels:", df["label"].unique())


Sequence columns: ['number_1', 'number_2', 'number_3', 'number_4', 'number_5', 'number_6', 'number_7', 'number_8', 'number_9', 'number_10']
Number of positions: 10
Labels: [0 1]


In [29]:

def extract_features(row):
    x = np.asarray(row, dtype=int)
    n = len(x)

    if n != 10:
        raise ValueError(f"Expected 10 numbers, got {n}.")

    # Frequency of unique values
    values, counts = np.unique(x, return_counts=True)
    frequencies = counts / n

    # Differences between adjacent numbers
    differences = np.diff(x)
    abs_differences = np.abs(differences)

    # Direction of each step: -1, 0, +1
    directions = np.sign(differences)
    nonzero_directions = directions[directions != 0]

    # Longest consecutive run of the same number
    longest_repeat_run = 1
    current_run = 1

    for i in range(1, n):
        if x[i] == x[i - 1]:
            current_run += 1
            longest_repeat_run = max(
                longest_repeat_run, current_run
            )
        else:
            current_run = 1

    # Longest strictly increasing/decreasing runs
    longest_increasing_run = 1
    longest_decreasing_run = 1
    current_inc = 1
    current_dec = 1

    for i in range(1, n):
        if x[i] > x[i - 1]:
            current_inc += 1
        else:
            current_inc = 1

        if x[i] < x[i - 1]:
            current_dec += 1
        else:
            current_dec = 1

        longest_increasing_run = max(
            longest_increasing_run, current_inc
        )
        longest_decreasing_run = max(
            longest_decreasing_run, current_dec
        )

    # Changes in direction, ignoring equal adjacent values
    direction_changes = int(
        np.sum(
            nonzero_directions[1:]
            != nonzero_directions[:-1]
        )
    )

    # Shannon entropy of the number distribution
    entropy = -np.sum(
        frequencies * np.log2(frequencies)
    )

    # Local peaks and valleys
    local_peaks = sum(
        x[i] > x[i - 1] and x[i] > x[i + 1]
        for i in range(1, n - 1)
    )

    local_valleys = sum(
        x[i] < x[i - 1] and x[i] < x[i + 1]
        for i in range(1, n - 1)
    )

    # Odd/even patterns
    parity = x % 2
    parity_changes = int(
        np.sum(parity[1:] != parity[:-1])
    )

    # Repetition in adjacent absolute gaps
    gap_values, gap_counts = np.unique(
        abs_differences, return_counts=True
    )

    # Ordered adjacent pairs
    pairs = list(zip(x[:-1], x[1:]))

    features = {
        # Basic statistics
        "unique_count": len(values),
        "repeat_count": n - len(values),
        "unique_ratio": len(values) / n,
        "max_frequency": counts.max(),
        "single_occurrence_count": int(np.sum(counts == 1)),
        "repeated_value_count": int(np.sum(counts > 1)),
        "mean": np.mean(x),
        "median": np.median(x),
        "std": np.std(x),
        "minimum": np.min(x),
        "maximum": np.max(x),
        "value_range": np.ptp(x),

        # Sequential features
        "adjacent_repeat_count": int(np.sum(differences == 0)),
        "longest_repeat_run": longest_repeat_run,
        "mean_adjacent_difference": np.mean(abs_differences),
        "std_adjacent_difference": np.std(abs_differences),
        "max_adjacent_difference": np.max(abs_differences),
        "ascending_pairs": int(np.sum(differences == 1)),
        "descending_pairs": int(np.sum(differences == -1)),
        "increasing_pairs": int(np.sum(differences > 0)),
        "decreasing_pairs": int(np.sum(differences < 0)),
        "small_gap_pairs": int(np.sum(abs_differences <= 2)),
        "longest_increasing_run": longest_increasing_run,
        "longest_decreasing_run": longest_decreasing_run,
        "direction_changes": direction_changes,

        # Distribution and randomness features
        "entropy": entropy,
        "odd_count": int(np.sum(parity == 1)),
        "even_count": int(np.sum(parity == 0)),
        "parity_changes": parity_changes,
        "lower_range_count": int(np.sum((x >= 1) & (x <= 5))),
        "upper_range_count": int(np.sum((x >= 6) & (x <= 10))),
        "gap_unique_count": len(gap_values),
        "gap_repeat_count": len(abs_differences) - len(gap_values),
        "max_gap_frequency": gap_counts.max(),
        "distinct_adjacent_pairs": len(set(pairs)),

        # Local patterns
        "local_peaks": int(local_peaks),
        "local_valleys": int(local_valleys),
    }

    # Frequency of each possible number, 1 through 10
    for number in range(1, 11):
        features[f"count_{number}"] = int(np.sum(x == number))

    return features


## 3. Build the feature matrix

Once the feature logic is defined, the notebook applies it across the full dataset and compares the engineered representation with the original feature set.

In [30]:

X_features = pd.DataFrame(
    [
        extract_features(row)
        for row in df[sequence_cols].to_numpy()
    ],
    index=df.index
)

y = df["label"].copy()

print("Original dataset shape:", df.shape)
print("Feature matrix shape:", X_features.shape)
print("Number of engineered features:", X_features.shape[1])

display(X_features.head())


Original dataset shape: (5848, 11)
Feature matrix shape: (5848, 47)
Number of engineered features: 47


,unique_count,repeat_count,unique_ratio,max_frequency,single_occurrence_count,repeated_value_count,mean,median,std,minimum,...,count_1,count_2,count_3,count_4,count_5,count_6,count_7,count_8,count_9,count_10
0,7,3,0.7,2,4,3,5.6,5.5,2.653300,2,...,0,2,1,1,1,1,0,2,2,0
1,9,1,0.9,2,8,1,4.7,4.5,2.609598,1,...,1,2,1,1,1,1,1,1,1,0
2,6,4,0.6,3,3,3,5.7,5.0,2.410394,1,...,1,0,0,2,3,0,1,1,2,0
3,9,1,0.9,2,8,1,4.6,4.5,2.727636,1,...,2,1,1,1,1,1,1,1,1,0
4,8,2,0.8,2,6,2,5.4,6.0,2.497999,1,...,1,1,1,0,1,2,2,1,1,0


### 3.1 Validate engineered features

Check for missing values, infinite values, constant columns, and duplicate feature columns before training.

In [31]:

# Missing values
print("Missing values:", X_features.isna().sum().sum())

# Infinite values
print(
    "Infinite values:",
    np.isinf(X_features.to_numpy(dtype=float)).sum()
)

# Constant features
constant_features = [
    col for col in X_features.columns
    if X_features[col].nunique() <= 1
]

print("\nConstant features:")
print(constant_features if constant_features else "None")

# Duplicate feature columns
duplicate_features = X_features.columns[
    X_features.T.duplicated()
].tolist()

print("\nDuplicate feature columns:")
print(duplicate_features if duplicate_features else "None")

# Summary statistics
display(X_features.describe().T)


Missing values: 0
Infinite values: 0

Constant features:
['count_10']

Duplicate feature columns:
None


,count,mean,std,min,25%,50%,75%,max
unique_count,5848.0,6.443912,1.128907,1.0,6.000000,6.000000,7.000000,9.000000
repeat_count,5848.0,3.556088,1.128907,1.0,3.000000,4.000000,4.000000,9.000000
unique_ratio,5848.0,0.644391,0.112891,0.1,0.600000,0.600000,0.700000,0.900000
max_frequency,5848.0,2.785568,0.761560,2.0,2.000000,3.000000,3.000000,10.000000
single_occurrence_count,5848.0,3.801129,1.736622,0.0,3.000000,4.000000,5.000000,8.000000
repeated_value_count,5848.0,2.642784,0.804126,1.0,2.000000,3.000000,3.000000,5.000000
mean,5848.0,4.966672,0.767353,1.0,4.500000,5.000000,5.400000,7.800000
median,5848.0,4.951094,1.189359,1.0,4.000000,5.000000,5.500000,9.000000
std,5848.0,2.429846,0.384146,0.0,2.193171,2.457641,2.690725,3.689173
minimum,5848.0,1.368331,0.680372,1.0,1.000000,1.000000,2.000000,7.000000


### 3.2 Compare features by label

Review how the engineered features differ between classes to understand which patterns are most informative.

In [32]:

feature_comparison = X_features.copy()
feature_comparison["label"] = y

comparison = feature_comparison.groupby("label").mean(
    numeric_only=True
).T

# Add difference between the two classes
if "Human" in comparison.columns and "Computer" in comparison.columns:
    comparison["absolute_mean_difference"] = (
        comparison["Human"] - comparison["Computer"]
    ).abs()

    comparison = comparison.sort_values(
        "absolute_mean_difference",
        ascending=False
    )

display(comparison)


label,0,1
unique_count,6.204800,7.853774
repeat_count,3.795200,2.146226
unique_ratio,0.620480,0.785377
max_frequency,2.882600,2.213443
single_occurrence_count,3.438600,5.938679
repeated_value_count,2.766200,1.915094
mean,4.994340,4.803538
median,4.987000,4.739387
std,2.420857,2.482847
minimum,1.417600,1.077830


### 3.3 Save the final dataset

Export the enriched feature table with the target label so it can be reused in modeling workflows.

In [33]:

engineered_df = X_features.copy()
engineered_df["label"] = y

output_path = data_dir / "engineered_features.csv"
engineered_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("Final dataset shape:", engineered_df.shape)


Saved: D:\personal projects\humanVsComputerGeneration\Data\engineered_features.csv
Final dataset shape: (5848, 48)
